
# Merge XGBoost LOO Joblib Files → Single Joblib + HDF5

This notebook scans a local `data/` folder for files matching `*_loo_xgboost_LOO_results.joblib`, loads each (expected to be a pandas DataFrame), adds a `protein` column from the filename if missing, merges them, and saves:
- `xgb_all_results.joblib` — merged `DataFrame` serialized with joblib
- `xgb_all_results.h5` — HDF5; prefers pandas HDFStore if PyTables is available; otherwise falls back to a generic HDF5 with CSV bytes

> If you see an error about missing `tables`, install PyTables (`pip install tables` or `conda install pytables`) and re-run the **Save HDF5** cell.


In [1]:
# Cell 1 — setup and helpers
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

import os, traceback
from pathlib import Path
import pandas as pd
import joblib

# Paths & settings
DATA_DIR = Path("data")
OUT_DIR  = Path("data_h5")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PARTS   = 3
OVERWRITE = False  # set True to regenerate existing .h5 files

def coerce_to_df(obj):
    if isinstance(obj, pd.DataFrame):
        return obj
    if hasattr(obj, "to_dataframe"):
        try:
            return obj.to_dataframe()
        except Exception:
            pass
    try:
        return pd.DataFrame(obj)
    except Exception:
        return pd.DataFrame({"value": [obj]})

def process_subset(subset_id, overwrite=False):
    """Slice files into N_PARTS and write this subset to HDF5 (format='fixed')."""
    all_files = sorted(DATA_DIR.glob("*.joblib"))
    subset = all_files[subset_id::N_PARTS]

    print(f"Subset {subset_id+1}/{N_PARTS}: {len(subset)} files")
    ok = err = 0
    for i, path in enumerate(subset, 1):
        out_path = OUT_DIR / (path.stem + ".h5")
        if out_path.exists() and not overwrite:
            print(f"[{i}/{len(subset)}] SKIP (exists): {path.name}")
            continue
        try:
            obj = joblib.load(path)
            df  = coerce_to_df(obj)
            df.to_hdf(out_path, key="data", mode="w", format="fixed")
            ok += 1
            print(f"[{i}/{len(subset)}] {path.name} -> {out_path.name} (rows={len(df)})")
        except Exception:
            err += 1
            print(f"[{i}/{len(subset)}] ERROR: {path.name}")
            traceback.print_exc()
    print(f"Done subset {subset_id+1}/{N_PARTS}. OK={ok}, ERR={err}")


convert directly to h5

In [2]:
# Cell 2 — run subset 1/3
process_subset(0, overwrite=OVERWRITE)
# Cell 3 — run subset 2/3
process_subset(1, overwrite=OVERWRITE) 
# Cell 4 — run subset 3/3   
process_subset(2, overwrite=OVERWRITE)

Subset 1/3: 458 files
[1/458] A2M_loo_xgboost_LOO_results.joblib -> A2M_loo_xgboost_LOO_results.h5 (rows=369)
[2/458] AAGAB_loo_xgboost_LOO_results.joblib -> AAGAB_loo_xgboost_LOO_results.h5 (rows=369)
[3/458] AAMP_loo_xgboost_LOO_results.joblib -> AAMP_loo_xgboost_LOO_results.h5 (rows=369)
[4/458] AARS_loo_xgboost_LOO_results.joblib -> AARS_loo_xgboost_LOO_results.h5 (rows=369)
[5/458] AASS_loo_xgboost_LOO_results.joblib -> AASS_loo_xgboost_LOO_results.h5 (rows=369)
[6/458] ABCB6_loo_xgboost_LOO_results.joblib -> ABCB6_loo_xgboost_LOO_results.h5 (rows=369)
[7/458] ABCC1_loo_xgboost_LOO_results.joblib -> ABCC1_loo_xgboost_LOO_results.h5 (rows=369)
[8/458] ABCD3_loo_xgboost_LOO_results.joblib -> ABCD3_loo_xgboost_LOO_results.h5 (rows=369)
[9/458] ABCF2_loo_xgboost_LOO_results.joblib -> ABCF2_loo_xgboost_LOO_results.h5 (rows=369)
[10/458] ABHD11_loo_xgboost_LOO_results.joblib -> ABHD11_loo_xgboost_LOO_results.h5 (rows=369)
[11/458] ABHD16A_loo_xgboost_LOO_results.joblib -> ABHD16A_loo_xg

Remove params columns to lighten the file size

In [ ]:
# Ultra-verbose HDF5 merge with diagnostics and a log file
import os, sys, time, platform, traceback
from pathlib import Path
from collections import defaultdict
import pandas as pd

# Optional: h5py for structural introspection when a file isn't a pandas store
try:
    import h5py
    H5PY_OK = True
except Exception:
    H5PY_OK = False

log_path = Path("merge_debug.log")

def log(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    try:
        with log_path.open("a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass

# --- banner / environment
log(">>> CELL START <<<")
log(f"python: {sys.version.splitlines()[0]}")
log(f"platform: {platform.platform()}")
log(f"pandas: {pd.__version__}")
try:
    import tables
    log(f"pytables: {tables.__version__}")
except Exception as e:
    log(f"pytables: NOT AVAILABLE ({e!r})")

# --- working dir + contents
cwd = Path().resolve()
log(f"cwd: {cwd}")
try:
    entries = [f"{p.name}/" if p.is_dir() else p.name for p in sorted(cwd.iterdir())]
    log("cwd contents: " + ", ".join(entries[:50]) + (" ..." if len(entries) > 50 else ""))
except Exception as e:
    log(f"cwd listing failed: {e!r}")

# --- find data_h5
candidates = [cwd / "data_h5", cwd / "model_results" / "data_h5"]
for i, p in enumerate(candidates):
    log(f"candidate[{i}]: {p} exists={p.exists()} is_dir={p.is_dir()}")

src_dir = next((p for p in candidates if p.is_dir()), None)
if src_dir is None:
    # try a broader search so we at least see where it might be
    hits = [p for p in cwd.rglob("data_h5") if p.is_dir()]
    log(f"rglob('data_h5') hits: {[str(h) for h in hits]}")
    raise FileNotFoundError("data_h5 folder not found from current working directory")

# --- list h5 files
files = sorted(src_dir.glob("*.h5"))
log(f"src_dir: {src_dir.resolve()}")
log(f"found {len(files)} .h5 files: {[f.name for f in files]}")
for f in files:
    try:
        size_mb = f.stat().st_size / (1024 * 1024)
        log(f"  {f.name} size={size_mb:.2f} MB")
    except Exception as e:
        log(f"  {f.name} size=? ({e!r})")

if not files:
    raise FileNotFoundError(f"No .h5 files found in {src_dir.resolve()}")

# --- inspect keys and collect data
frames_by_key = defaultdict(list)
total_rows_seen = 0
files_processed = 0

for f in files:
    log(f"OPEN: {f.name}")
    try:
        with pd.HDFStore(f, mode="r") as store:
            keys = list(store.keys())
            log(f"  pandas store keys: {keys}")
            if not keys:
                log("  (no keys)")
            for key in keys:
                try:
                    obj = store.get(key)
                    typ = type(obj).__name__
                    log(f"    GET {key} -> {typ}")
                    if isinstance(obj, pd.Series):
                        obj = obj.to_frame()
                        typ = "DataFrame(from Series)"
                    if isinstance(obj, pd.DataFrame):
                        df = obj.copy()
                        df["__source_file__"] = f.name
                        frames_by_key[key].append(df)
                        total_rows_seen += len(df)
                        log(f"      appended rows={len(df)} cols={df.shape[1]}")
                    else:
                        log("      (not a DataFrame/Series; skipping)")
                except Exception as e_key:
                    log(f"    ERROR reading key {key} from {f.name}: {e_key!r}")
        files_processed += 1
    except Exception as e_pd:
        log(f"  pandas HDFStore failed: {e_pd!r}")
        if H5PY_OK:
            try:
                with h5py.File(f, "r") as h5:
                    def walk(name, obj):
                        kind = "Group" if isinstance(obj, h5py.Group) else "Dataset"
                        log(f"    h5py {kind}: /{name} shape={getattr(obj, 'shape', None)} dtype={getattr(obj, 'dtype', None)}")
                    h5.visititems(walk)
            except Exception as e_h5:
                log(f"  h5py open failed: {e_h5!r}")

log(f"files processed (as pandas stores): {files_processed}")
log(f"frames_by_key keys: {list(frames_by_key.keys())}")
log(f"total_rows_seen (pre-merge): {total_rows_seen}")

if not frames_by_key:
    raise ValueError(
        "No pandas DataFrame/Series objects found in these HDF5 files. "
        "If the files are raw h5py datasets (not pandas stores), we switch strategy."
    )

# --- merge + write
out_path = src_dir.parent / "merged.h5"
log(f"OUTPUT path: {out_path.resolve()} (will overwrite)")
try:
    with pd.HDFStore(out_path, mode="w") as out:
        for key, parts in frames_by_key.items():
            log(f"MERGE key={key} with {len(parts)} part(s)")
            merged = pd.concat(parts, ignore_index=True, sort=False)
            log(f"  merged rows={len(merged)} cols={merged.shape[1]}")
            out.put(key, merged, format="table", data_columns=True)
            log(f"  wrote key={key}")
    log(">>> DONE OK <<<")
except Exception as e_out:
    log(f"WRITE FAILED: {e_out!r}")
    traceback.print_exc()
    raise


In [7]:
# Merge all per-protein HDF5 files into a single HDF5 (robust)
from pathlib import Path
import pandas as pd
import numpy as np
import traceback
import warnings
from pandas.errors import PerformanceWarning
import gc

# Quiet PyTables' object-pickling warnings (object columns are expected)
warnings.filterwarnings("ignore", category=PerformanceWarning)

DATA_H5  = Path("data_h5")
OUT_PATH = Path("xgb_all_results2.h5")   # change if we want a different location/name
PREFERRED_KEY = "data"           # we try this first; fall back to the first key if missing

# Optional: keep only specific columns to reduce memory (set to a list or leave as None)
KEEP_ONLY = None
# e.g. KEEP_ONLY = ["protein", "LOO sample", "true", "predicted"]

files = sorted(DATA_H5.glob("*.h5"))
print(f"Found {len(files)} h5 files in {DATA_H5}")

frames = []
all_cols = []   # running union of column names (keeps stable order)

def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure string, unique, flat column names (handles MultiIndex, duplicates)."""
    cols = df.columns
    if isinstance(cols, pd.MultiIndex):
        new_cols = ["__".join(map(str, tup)) for tup in cols.to_list()]
    else:
        new_cols = [str(c) for c in cols.to_list()]

    # De-duplicate while preserving order (suffix with .1, .2, ...)
    seen = {}
    uniq_cols = []
    for c in new_cols:
        if c not in seen:
            seen[c] = 0
            uniq_cols.append(c)
        else:
            seen[c] += 1
            uniq_cols.append(f"{c}.{seen[c]}")
    df.columns = uniq_cols
    return df

def _coerce_keep_only(df: pd.DataFrame, keep):
    if keep is None:
        return df
    # Keep the intersection and preserve keep order
    safe_keep = [c for c in keep if c in df.columns]
    return df.loc[:, safe_keep] if safe_keep else df.iloc[:, 0:0]  # empty frame with 0 cols if none matched

def _update_union_columns(current_union, new_cols):
    """Append columns not yet in union, preserving order."""
    for c in new_cols:
        if c not in current_union:
            current_union.append(c)

# Load all frames, being tolerant to missing keys or odd structures
for i, fp in enumerate(files, 1):
    try:
        # Open store, pick a key
        with pd.HDFStore(fp, mode="r") as store:
            keys = list(store.keys())
            if not keys:
                print(f"[{i}/{len(files)}] WARNING: {fp.name} contains no keys; skipping.")
                continue

            if f"/{PREFERRED_KEY}" in keys:
                key_to_use = PREFERRED_KEY
            else:
                # Fall back to the first available key (strip the leading "/")
                key_to_use = keys[0].lstrip("/")
                if key_to_use != PREFERRED_KEY:
                    print(f"[{i}/{len(files)}] INFO: {fp.name} missing key '{PREFERRED_KEY}', using '{key_to_use}'")

        # Now actually read the selected key
        df = pd.read_hdf(fp, key=key_to_use)

        # Ensure flat string columns and no dupes
        df = _flatten_columns(df)

        # Normalize index (avoid concat alignment surprises)
        df = df.reset_index(drop=True)

        # Optionally reduce columns early
        df = _coerce_keep_only(df, KEEP_ONLY)

        # Track union of columns to align later
        _update_union_columns(all_cols, list(df.columns))

        frames.append(df)

        if i % 50 == 0 or i == len(files):
            print(f"[{i}/{len(files)}] loaded: {fp.name} (rows={len(df)}, cols={df.shape[1]})")

    except (OSError, ValueError, KeyError) as e:
        print(f"[{i}/{len(files)}] ERROR reading {fp.name}: {e}")
        traceback.print_exc()
    except Exception as e:
        print(f"[{i}/{len(files)}] UNEXPECTED ERROR reading {fp.name}: {e}")
        traceback.print_exc()

if not frames:
    raise RuntimeError("No data frames loaded — nothing to merge.")

# Align columns across all frames (outer-join semantics with NaNs for missing)
aligned = []
for idx, f in enumerate(frames, 1):
    # Reindex to union; keep types as-is (pandas will upcast as needed)
    af = f.reindex(columns=all_cols)
    aligned.append(af)
    # Periodic GC for large merges
    if idx % 100 == 0:
        gc.collect()

# Concatenate
merged = pd.concat(aligned, axis=0, ignore_index=True, sort=False, copy=False)
print(f"Merged shape: {merged.shape}")

# Write as 'fixed' (pickle-friendly; supports object columns)
# If this ever raises due to a pathological object type, uncomment the astype('object') fallback below.
# merged = merged.astype('object')
merged.to_hdf(OUT_PATH, key=PREFERRED_KEY, mode="w", format="fixed")
print(f"Wrote: {OUT_PATH.resolve()}  (rows={len(merged)}, cols={merged.shape[1]})")


Found 1373 h5 files in data_h5
[50/1373] loaded: ACBD5_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[100/1373] loaded: ADNP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[150/1373] loaded: AKAP8L_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[200/1373] loaded: ANKLE2_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[250/1373] loaded: APP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[300/1373] loaded: ARL3_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[350/1373] loaded: ATF7IP_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[400/1373] loaded: ATPAF1_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[450/1373] loaded: BCL2L13_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[500/1373] loaded: BST2_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[550/1373] loaded: C3orf17_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[600/1373] loaded: CARKD_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[650/1373] loaded: CCDC50_loo_xgboost_LOO_results.h5 (rows=369, cols=4)
[700/1373] loaded: CDC20_loo_xgboost_LOO_re

In [8]:
# Open the merged HDF5 file and display its head + NaN counts for true/predicted
from pathlib import Path
import pandas as pd
from IPython.display import display

h5_path = Path("xgb_all_results2.h5")
preferred_key = "data"

if not h5_path.exists():
    raise FileNotFoundError(f"{h5_path.resolve()} not found")

with pd.HDFStore(h5_path, mode="r") as store:
    keys = list(store.keys())
    if not keys:
        raise RuntimeError(f"No tables found in {h5_path}")
    use_key = f"/{preferred_key}" if f"/{preferred_key}" in keys else keys[0]
    if use_key != f"/{preferred_key}":
        print(f"Info: key '{preferred_key}' not found. Using '{use_key.lstrip('/')}'")

df = pd.read_hdf(h5_path, key=use_key.lstrip("/"))
display(df.head())
print(f"Rows={len(df)}, Cols={df.shape[1]}, Key='{use_key.lstrip('/')}'")

# NaN counts (and percentages) in true and predicted
if {"true", "predicted"}.issubset(df.columns):
    n_true_nan = int(df["true"].isna().sum())
    n_pred_nan = int(df["predicted"].isna().sum())
    n_either_nan = int(df[["true","predicted"]].isna().any(axis=1).sum())
    total = len(df) if len(df) > 0 else 1
    print(
        f"NaNs — true: {n_true_nan} ({n_true_nan/total:.2%}), "
        f"predicted: {n_pred_nan} ({n_pred_nan/total:.2%}), "
        f"either: {n_either_nan} ({n_either_nan/total:.2%})"
    )
else:
    print("Columns 'true' and/or 'predicted' not found. Available columns:", list(df.columns))


,protein,LOO sample,true,predicted
0,A2M,MDAMB468_BREAST,-0.345562,-0.602695
1,A2M,SH4_SKIN,1.642186,0.980278
2,A2M,AU565_BREAST,-1.129114,-0.371104
3,A2M,KMRC1_KIDNEY,1.493333,0.870634
4,A2M,CAL51_BREAST,0.274652,0.220913


Rows=506637, Cols=4, Key='data'
NaNs — true: 0 (0.00%), predicted: 0 (0.00%), either: 0 (0.00%)


In [15]:
# Cell — compare proteins in file vs data (handles your path & formatting)
from pathlib import Path
import pandas as pd

# Build set of proteins in your loaded data (normalize whitespace & case)
assert "protein" in df.columns, "Expected column 'protein' in df"
proteins_in_data = (
    df["protein"]
    .dropna().astype(str).str.strip().str.upper()
    .unique().tolist()
)
proteins_in_data = set(proteins_in_data)

# Try common locations; picks the first that exists
candidates = [
    Path("protein_list.txt"),
    Path("model_results") / "protein_list.txt",
    Path("04. xgb") / "model_results" / "protein_list.txt",  # from your screenshot
]
for cand in candidates:
    if cand.exists():
        list_path = cand
        break
else:
    raise FileNotFoundError(
        "protein_list.txt not found in any of: "
        + ", ".join(str(p) for p in candidates)
    )

# Read file: ignore blank lines and lines starting with '#'
with list_path.open("r", encoding="utf-8") as f:
    file_lines = [ln.strip() for ln in f]

file_series = (
    pd.Series(file_lines, dtype="string")
      .str.strip()
      .pipe(lambda s: s[~s.str.startswith("#", na=False)])
      .replace("", pd.NA)
      .dropna()
      .str.upper()
      .drop_duplicates()
)

file_proteins = set(file_series.tolist())

missing = sorted(file_proteins - proteins_in_data)   # in file but NOT in data
extras  = sorted(proteins_in_data - file_proteins)   # in data but NOT in file
overlap = sorted(file_proteins & proteins_in_data)

print(f"File: {list_path}  |  total in file (unique): {len(file_proteins)}")
print(f"In file BUT NOT in data: {len(missing)}")
print(missing[:50])  # preview
print(f"\nIn data BUT NOT in file: {len(extras)}")
print(extras[:50])   # preview
print(f"\nOverlap: {len(overlap)}")

print("\nSaved: proteins_not_in_all_data.txt, proteins_in_data_not_in_list.txt")


Part 2/3: 55 files
[50/55] ADRM1_loo_xgboost_LOO_results.joblib -> shape=(369, 4)
[55/55] AFG3L2_loo_xgboost_LOO_results.joblib -> shape=(369, 4)
Part 2 merged shape: (20295, 4)
Wrote joblib: c:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\04. xgb\model_results\xgb_results_part2.joblib


In [1]:
from pathlib import Path
import pandas as pd

def merge_h5_folder(
    input_folder="final_data",
    output_folder="xgb_res",
    output_file="xgb_all_results.h5",
    key=None,
    output_key="data",
    add_source_column=False,
    save=True
):
    """
    Merge all .h5 files from input_folder into one DataFrame and save it to output_folder.

    Parameters
    ----------
    input_folder : str or Path
        Folder containing source .h5 files.
    output_folder : str or Path
        Folder where the merged file will be saved.
    output_file : str
        Name of the merged output file.
    key : str or None
        HDF5 key to read. If None, each file must contain exactly one key.
    output_key : str
        Key used when saving the merged file.
    add_source_column : bool
        If True, adds a column with the source filename.
    save : bool
        If True, saves the merged DataFrame.

    Returns
    -------
    pd.DataFrame
        The merged DataFrame.
    """
    input_folder = Path(input_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    h5_files = sorted(input_folder.glob("*.h5"))

    if not h5_files:
        raise FileNotFoundError(f"No .h5 files found in: {input_folder}")

    dfs = []

    for file in h5_files:
        with pd.HDFStore(file, mode="r") as store:
            keys = store.keys()

        if not keys:
            print(f"Skipping empty file: {file.name}")
            continue

        if key is None:
            if len(keys) != 1:
                raise ValueError(
                    f"{file.name} has multiple keys {keys}. Pass key='your_key' explicitly."
                )
            read_key = keys[0]
        else:
            read_key = key if key.startswith("/") else f"/{key}"
            if read_key not in keys:
                raise KeyError(f"Key {read_key} not found in {file.name}. Available: {keys}")

        df = pd.read_hdf(file, key=read_key)

        if add_source_column:
            df = df.copy()
            df["__source_file__"] = file.name

        dfs.append(df)

    if not dfs:
        raise ValueError("No readable data found in the .h5 files.")

    merged = pd.concat(dfs, ignore_index=True)

    if save:
        output_path = output_folder / output_file
        merged.to_hdf(output_path, key=output_key, mode="w")
        print(f"Merged file saved to: {output_path}")

    return merged

In [2]:
merged_df = merge_h5_folder(
    input_folder="final_data",
    output_folder="xgb_res",
    output_file="xgb_all_results.h5"
)

Merged file saved to: xgb_res\xgb_all_results.h5
